In [1]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

I0000 00:00:1779975427.714357   17263 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779975429.950106   17263 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
folder_path = '/mnt/d/Main/Core/DL/Starting Deep Learning/Deep_Learning/Deep Learning/CNN/Keras Functional Model/UTKface/UTKFace'

In [3]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [4]:
len(age)

23708

In [5]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [6]:
df.shape

(23708, 3)

In [7]:
df.head()

,age,gender,img
0,100,0,100_0_0_20170112213500903.jpg.chip.jpg
1,100,0,100_0_0_20170112215240346.jpg.chip.jpg
2,100,1,100_1_0_20170110183726390.jpg.chip.jpg
3,100,1,100_1_0_20170112213001988.jpg.chip.jpg
4,100,1,100_1_0_20170112213303693.jpg.chip.jpg


In [8]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [9]:
test_df.shape

(3708, 3)

In [10]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [11]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='raw')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='raw')


Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [12]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [13]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [14]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [15]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [16]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [17]:
def multi_output_generator(generator):
    while True:
        x, y = next(generator)
        yield x, {'age': y[:, 0], 'gender': y[:, 1]}

train_multi_generator = multi_output_generator(train_generator)
test_multi_generator = multi_output_generator(test_generator)

model.fit(train_multi_generator,
          steps_per_epoch=len(train_generator),
          epochs=10,
          validation_data=test_multi_generator,
          validation_steps=len(test_generator))


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 173s 261ms/step - age_loss: 15.4036 - age_mae: 15.4036 - gender_accuracy: 0.5073 - gender_loss: 0.8734 - loss: 101.8673 - val_age_loss: 16.4998 - val_age_mae: 16.4965 - val_gender_accuracy: 0.5173 - val_gender_loss: 0.6926 - val_loss: 85.0616
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 162s 260ms/step - age_loss: 14.9915 - age_mae: 14.9915 - gender_accuracy: 0.5225 - gender_loss: 0.6988 - loss: 84.1719 - val_age_loss: 14.9405 - val_age_mae: 14.9408 - val_gender_accuracy: 0.5173 - val_gender_loss: 0.6928 - val_loss: 83.5239
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 176s 282ms/step - age_loss: 14.8844 - age_mae: 14.8844 - gender_accuracy: 0.5235 - gender_loss: 0.6923 - loss: 83.4260 - val_age_loss: 14.7668 - val_age_mae: 14.7715 - val_gender_accuracy: 0.5183 - val_gender_loss: 0.6925 - val_loss: 83.3265
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 194s 310ms/step - age_loss: 14.8326 - age_mae: 14.8326 - gender_accuracy: 0.5239 - gender_loss: 0.6945 - loss: 83